In [2]:
# ============================================================
# Cell 1: Setup and Mount Drive
# ============================================================
"""
VGGSound Dataset Download and Extraction
This notebook downloads a subset of VGGSound from HuggingFace,
extracts audio and frames, and saves to Google Drive.
"""

from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Create base directory
BASE_DIR = "/content/drive/MyDrive/spectralbridge_data"
os.makedirs(BASE_DIR, exist_ok=True)
os.makedirs(f"{BASE_DIR}/raw/vggsound", exist_ok=True)
os.makedirs(f"{BASE_DIR}/processed/audio", exist_ok=True)
os.makedirs(f"{BASE_DIR}/processed/images", exist_ok=True)

print(f"✓ Base directory created: {BASE_DIR}")
print(f"✓ Free space: {os.statvfs(BASE_DIR).f_bavail * os.statvfs(BASE_DIR).f_frsize / (1024**3):.1f} GB")



Mounted at /content/drive
✓ Base directory created: /content/drive/MyDrive/spectralbridge_data
✓ Free space: 182.6 GB


In [4]:
# ============================================================
# Cell 2: Install Dependencies
# ============================================================
!pip install -q datasets ffmpeg-python
!apt-get install -y ffmpeg > /dev/null 2>&1

import ffmpeg
from datasets import load_dataset
import tarfile
from pathlib import Path
from tqdm.notebook import tqdm
import shutil

print("✓ Dependencies installed")

✓ Dependencies installed


In [5]:
# ============================================================
# Cell 3: Selected Categories (CORRECTED NAMES)
# ============================================================
"""
Selected VGGSound categories for SpectralBridge.
Mix of textually-easy and perceptually-rich categories.
"""

SELECTED_CATEGORIES = [
    # Textually easy (clear semantic descriptions)
    "playing piano",
    "playing acoustic guitar",
    "dog barking",
    "cat meowing",
    "helicopter",
    "typing on computer keyboard",
    "telephone bell ringing",

    # Perceptually rich (atmospheric/textural qualities)
    "fire crackling",
    "ocean burbling",
    "thunder",
    "raining",
    "wind noise",
    "wind rustling leaves",
    "waterfall burbling",
    "hammering nails",
    "stream burbling",
]

print(f"Selected {len(SELECTED_CATEGORIES)} categories:")
for cat in SELECTED_CATEGORIES:
    print(f"  - {cat}")

Selected 16 categories:
  - playing piano
  - playing acoustic guitar
  - dog barking
  - cat meowing
  - helicopter
  - typing on computer keyboard
  - telephone bell ringing
  - fire crackling
  - ocean burbling
  - thunder
  - raining
  - wind noise
  - wind rustling leaves
  - waterfall burbling
  - hammering nails
  - stream burbling


In [4]:
# ============================================================
# Cell 4: Download with HTTP Range Resume
# ============================================================
import requests
import tarfile
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm
import shutil

def download_with_resume(url, filepath, max_retries=5):
    """Download large file with resume capability using HTTP Range headers."""

    filepath = Path(filepath)
    temp_filepath = filepath.with_suffix('.tmp')

    # Check if we have a partial download
    if temp_filepath.exists():
        resume_byte_pos = temp_filepath.stat().st_size
        print(f"  Resuming from {resume_byte_pos / (1024**3):.2f} GB")
    else:
        resume_byte_pos = 0

    for attempt in range(max_retries):
        try:
            # Request with Range header for resume
            headers = {'Range': f'bytes={resume_byte_pos}-'} if resume_byte_pos > 0 else {}

            response = requests.get(url, headers=headers, stream=True, timeout=120)

            # Check if server supports resume
            if resume_byte_pos > 0 and response.status_code not in [200, 206]:
                print(f"  Server doesn't support resume, restarting download...")
                resume_byte_pos = 0
                temp_filepath.unlink(missing_ok=True)
                continue

            total_size = int(response.headers.get('content-length', 0))

            # Open file for appending if resuming, otherwise write new
            mode = 'ab' if resume_byte_pos > 0 else 'wb'

            with open(temp_filepath, mode) as f:
                with tqdm(
                    total=total_size + resume_byte_pos,
                    initial=resume_byte_pos,
                    unit='B',
                    unit_scale=True,
                    desc=filepath.name
                ) as pbar:
                    for chunk in response.iter_content(chunk_size=8192):
                        if chunk:
                            f.write(chunk)
                            pbar.update(len(chunk))

            # Download complete - rename temp file
            temp_filepath.rename(filepath)
            return True

        except Exception as e:
            print(f"  Attempt {attempt + 1}/{max_retries} failed: {str(e)[:80]}")
            if attempt < max_retries - 1:
                print(f"  Retrying in 10 seconds...")
                import time
                time.sleep(10)
                # Update resume position if partial download exists
                if temp_filepath.exists():
                    resume_byte_pos = temp_filepath.stat().st_size
            else:
                print(f"  All {max_retries} attempts failed")
                return False

    return False

def download_and_extract_tar_files_resumable(start_from=2, num_files=1):
    """Download with proper resume support."""

    print(f"\n{'='*60}")
    print(f"Downloading VGGSound tar files (with resume)")
    print(f"{'='*60}\n")

    # Load CSV
    csv_url = "https://huggingface.co/datasets/Loie/VGGSound/resolve/main/vggsound.csv"
    csv_path = "/content/vggsound.csv"

    if not Path(csv_path).exists():
        print("Downloading vggsound.csv...")
        response = requests.get(csv_url)
        with open(csv_path, 'wb') as f:
            f.write(response.content)

    vgg_df = pd.read_csv(csv_path, header=None, names=['video_id', 'start_time', 'label', 'split'])
    print(f"✓ Loaded {len(vgg_df)} videos")

    # Build lookup
    video_to_category = {}
    for idx, row in vgg_df.iterrows():
        if row['label'] in SELECTED_CATEGORIES:
            key = f"{row['video_id']}_{int(row['start_time']):06d}"
            video_to_category[key] = row['label']

    print(f"✓ Found {len(video_to_category)} videos in our categories\n")

    # Count existing
    raw_dir = Path(BASE_DIR) / "raw" / "vggsound"
    category_counts = {cat: 0 for cat in SELECTED_CATEGORIES}

    if raw_dir.exists():
        for cat in SELECTED_CATEGORIES:
            cat_dir = raw_dir / cat
            if cat_dir.exists():
                category_counts[cat] = len(list(cat_dir.glob("*.mp4")))

        total = sum(category_counts.values())
        if total > 0:
            print(f"Already have {total} videos:")
            for cat, count in category_counts.items():
                if count > 0:
                    print(f"  {cat:30s}: {count}")
            print()

    base_url = "https://huggingface.co/datasets/Loie/VGGSound/resolve/main"
    temp_dir = Path("/content/vggsound_temp")
    temp_dir.mkdir(exist_ok=True)

    # Process tar files
    for i in range(start_from, start_from + num_files):
        tar_name = f"vggsound_{i:02d}.tar.gz"
        tar_path = temp_dir / tar_name

        print(f"\n{'='*60}")
        print(f"Processing {tar_name}")
        print(f"{'='*60}\n")

        url = f"{base_url}/{tar_name}"

        # Download with resume support
        print(f"Downloading with resume capability...")
        success = download_with_resume(url, tar_path, max_retries=5)

        if not success:
            print(f"✗ Failed to download {tar_name} after all retries")
            print(f"  Skipping to next file...\n")
            continue

        print(f"✓ Downloaded successfully\n")

        # Extract
        print("Extracting...")
        new_count = 0

        try:
            with tarfile.open(tar_path, 'r:gz') as tar:
                members = tar.getmembers()

                for member in tqdm(members, desc="Processing"):
                    if not member.name.endswith('.mp4'):
                        continue

                    key = Path(member.name).stem
                    category = video_to_category.get(key)

                    if not category or category_counts[category] >= 80:
                        continue

                    cat_dir = raw_dir / category
                    cat_dir.mkdir(parents=True, exist_ok=True)

                    output_path = cat_dir / f"{key}.mp4"

                    if output_path.exists():
                        continue

                    file_obj = tar.extractfile(member)
                    if file_obj:
                        output_path.write_bytes(file_obj.read())
                        category_counts[category] += 1
                        new_count += 1

            print(f"\n✓ Extracted {new_count} new videos")

            # Clean up downloaded tar
            tar_path.unlink()
            # Also clean up any .tmp files
            for tmp_file in temp_dir.glob("*.tmp"):
                tmp_file.unlink()

        except Exception as e:
            print(f"✗ Extraction error: {e}")
            continue

        print("\nCurrent totals:")
        for cat, count in sorted(category_counts.items()):
            if count > 0:
                print(f"  {cat:30s}: {count}")

        if all(count >= 50 for count in category_counts.values()):
            print("\n✓ Target reached!")
            break

    print(f"\n{'='*60}")
    print("DONE")
    print(f"{'='*60}")
    for cat, count in sorted(category_counts.items()):
        print(f"{cat:30s}: {count}")

    shutil.rmtree(temp_dir, ignore_errors=True)
    return category_counts

In [5]:
# ============================================================
# Cell 5: Extract Audio and Frame Function
# ============================================================
def extract_audio_and_frame(mp4_path, output_audio_path, output_image_path):
    """
    Extract audio as WAV and middle frame as JPG from MP4.

    Args:
        mp4_path: Path to input MP4
        output_audio_path: Path for output WAV file
        output_image_path: Path for output JPG file
    """
    try:
        # Get video duration to find middle frame
        probe = ffmpeg.probe(str(mp4_path))
        duration = float(probe['streams'][0]['duration'])
        middle_time = duration / 2

        # Extract audio (16kHz mono WAV)
        (
            ffmpeg
            .input(str(mp4_path))
            .output(str(output_audio_path),
                   acodec='pcm_s16le',
                   ac=1,
                   ar='16000')
            .overwrite_output()
            .run(quiet=True)
        )

        # Extract middle frame (518x518 for DINOv2)
        (
            ffmpeg
            .input(str(mp4_path), ss=middle_time)
            .filter('scale', 518, 518)
            .output(str(output_image_path), vframes=1)
            .overwrite_output()
            .run(quiet=True)
        )

        return True
    except Exception as e:
        print(f"Error processing {mp4_path}: {e}")
        return False

In [6]:
# ============================================================
# Cell 6: Process All Videos
# ============================================================
def process_all_videos():
    """
    Process all downloaded MP4s: extract audio and frames.
    """
    print(f"\n{'='*60}")
    print("Processing all videos...")
    print(f"{'='*60}\n")

    raw_dir = Path(BASE_DIR) / "raw" / "vggsound"
    audio_dir = Path(BASE_DIR) / "processed" / "audio"
    image_dir = Path(BASE_DIR) / "processed" / "images"

    stats = {"success": 0, "failed": 0}

    # Process each category
    for category in SELECTED_CATEGORIES:
        category_raw = raw_dir / category

        if not category_raw.exists():
            continue

        # Create output directories
        (audio_dir / category).mkdir(parents=True, exist_ok=True)
        (image_dir / category).mkdir(parents=True, exist_ok=True)

        # Process each MP4
        mp4_files = list(category_raw.glob("*.mp4"))

        print(f"\nProcessing {category}: {len(mp4_files)} videos")

        for mp4_path in tqdm(mp4_files, desc=category):
            video_id = mp4_path.stem

            audio_path = audio_dir / category / f"{video_id}.wav"
            image_path = image_dir / category / f"{video_id}.jpg"

            # Skip if already processed
            if audio_path.exists() and image_path.exists():
                stats["success"] += 1
                continue

            # Extract
            success = extract_audio_and_frame(mp4_path, audio_path, image_path)

            if success:
                stats["success"] += 1
            else:
                stats["failed"] += 1

    print(f"\n{'='*60}")
    print("Processing Summary:")
    print(f"{'='*60}")
    print(f"Successful: {stats['success']}")
    print(f"Failed: {stats['failed']}")
    print(f"Success rate: {stats['success']/(stats['success']+stats['failed'])*100:.1f}%")

    return stats

In [45]:
# ============================================================
# Cell 7: Re-extract with Correct Category Names
# ============================================================
print("Re-extracting from already downloaded tar files...")
print("Using CORRECTED category names (with spaces!)\n")

# This will check the tar files we downloaded and extract with correct matching
category_counts = download_and_extract_tar_files_resumable(start_from=2, num_files=6)

Re-extracting from already downloaded tar files...
Using CORRECTED category names (with spaces!)



✓ Loaded 199467 videos
✓ Found 10841 videos in our categories


Processing vggsound_02.tar.gz



vggsound_02.tar.gz:   0%|          | 0.00/16.3G [00:00<?, ?B/s]

✓ Downloaded successfully

Extracting...


Processing:   0%|          | 0/10000 [00:00<?, ?it/s]


✓ Extracted 599 new videos

Current totals:
  cat meowing                   : 75
  dog barking                   : 42
  fire crackling                : 28
  hammering nails               : 26
  helicopter                    : 40
  ocean burbling                : 43
  playing acoustic guitar       : 80
  playing piano                 : 40
  raining                       : 28
  stream burbling               : 36
  telephone bell ringing        : 9
  thunder                       : 7
  typing on computer keyboard   : 23
  waterfall burbling            : 26
  wind noise                    : 65
  wind rustling leaves          : 31

Processing vggsound_03.tar.gz



...


✓ Extracted 34 new videos

Current totals:
  cat meowing                   : 80
  dog barking                   : 80
  fire crackling                : 80
  hammering nails               : 80
  helicopter                    : 80
  ocean burbling                : 80
  playing acoustic guitar       : 80
  playing piano                 : 80
  raining                       : 80
  stream burbling               : 80
  telephone bell ringing        : 39
  thunder                       : 53
  typing on computer keyboard   : 80
  waterfall burbling            : 80
  wind noise                    : 80
  wind rustling leaves          : 80

Processing vggsound_06.tar.gz



vggsound_06.tar.gz:   0%|          | 0.00/17.2G [00:00<?, ?B/s]

✓ Downloaded successfully

Extracting...


Processing:   0%|          | 0/10000 [00:00<?, ?it/s]


✓ Extracted 26 new videos

Current totals:
  cat meowing                   : 80
  dog barking                   : 80
  fire crackling                : 80
  hammering nails               : 80
  helicopter                    : 80
  ocean burbling                : 80
  playing acoustic guitar       : 80
  playing piano                 : 80
  raining                       : 80
  stream burbling               : 80
  telephone bell ringing        : 55
  thunder                       : 63
  typing on computer keyboard   : 80
  waterfall burbling            : 80
  wind noise                    : 80
  wind rustling leaves          : 80

✓ Target reached!

DONE
cat meowing                   : 80
dog barking                   : 80
fire crackling                : 80
hammering nails               : 80
helicopter                    : 80
ocean burbling                : 80
playing acoustic guitar       : 80
playing piano                 : 80
raining                       : 80
stream burbling         

In [46]:
# Get remaining videos for telephone and thunder
category_counts = download_and_extract_tar_files_resumable(start_from=7, num_files=1)



✓ Loaded 199467 videos
✓ Found 10841 videos in our categories

Already have 1238 videos:
  playing piano                 : 80
  playing acoustic guitar       : 80
  dog barking                   : 80
  cat meowing                   : 80
  helicopter                    : 80
  typing on computer keyboard   : 80
  telephone bell ringing        : 55
  fire crackling                : 80
  ocean burbling                : 80
  thunder                       : 63
  raining                       : 80
  wind noise                    : 80
  wind rustling leaves          : 80
  waterfall burbling            : 80
  hammering nails               : 80
  stream burbling               : 80


Processing vggsound_07.tar.gz



vggsound_07.tar.gz:   0%|          | 0.00/17.1G [00:00<?, ?B/s]

  Attempt 1/5 failed: ('Connection broken: IncompleteRead(16972816057 bytes read, 164291862 more expec
  Retrying in 10 seconds...


vggsound_07.tar.gz:  99%|#########9| 17.0G/17.1G [00:00<?, ?B/s]

✓ Downloaded successfully

Extracting...


Processing:   0%|          | 0/10000 [00:00<?, ?it/s]


✓ Extracted 31 new videos

Current totals:
  cat meowing                   : 80
  dog barking                   : 80
  fire crackling                : 80
  hammering nails               : 80
  helicopter                    : 80
  ocean burbling                : 80
  playing acoustic guitar       : 80
  playing piano                 : 80
  raining                       : 80
  stream burbling               : 80
  telephone bell ringing        : 70
  thunder                       : 79
  typing on computer keyboard   : 80
  waterfall burbling            : 80
  wind noise                    : 80
  wind rustling leaves          : 80

✓ Target reached!

DONE
cat meowing                   : 80
dog barking                   : 80
fire crackling                : 80
hammering nails               : 80
helicopter                    : 80
ocean burbling                : 80
playing acoustic guitar       : 80
playing piano                 : 80
raining                       : 80
stream burbling         

In [7]:
# ============================================================
# Cell 8: Extract Audio and Images from MP4s
# ============================================================
import ffmpeg
from pathlib import Path
from tqdm.notebook import tqdm

def extract_audio_and_frame(mp4_path, output_audio_path, output_image_path):
    """
    Extract audio as WAV and middle frame as JPG from MP4.

    Args:
        mp4_path: Path to input MP4
        output_audio_path: Path for output WAV file
        output_image_path: Path for output JPG file
    """
    try:
        # Get video duration to find middle frame
        probe = ffmpeg.probe(str(mp4_path))
        duration = float(probe['streams'][0]['duration'])
        middle_time = duration / 2

        # Extract audio (16kHz mono WAV)
        (
            ffmpeg
            .input(str(mp4_path))
            .output(str(output_audio_path),
                   acodec='pcm_s16le',
                   ac=1,
                   ar='16000')
            .overwrite_output()
            .run(quiet=True, capture_stderr=True)
        )

        # Extract middle frame (518x518 for DINOv2)
        (
            ffmpeg
            .input(str(mp4_path), ss=middle_time)
            .filter('scale', 518, 518)
            .output(str(output_image_path), vframes=1)
            .overwrite_output()
            .run(quiet=True, capture_stderr=True)
        )

        return True
    except Exception as e:
        return False

def process_all_videos():
    """
    Process all downloaded MP4s: extract audio and frames.
    """
    print(f"\n{'='*60}")
    print("Processing all videos...")
    print(f"{'='*60}\n")

    raw_dir = Path(BASE_DIR) / "raw" / "vggsound"
    audio_dir = Path(BASE_DIR) / "processed" / "audio"
    image_dir = Path(BASE_DIR) / "processed" / "images"

    stats = {"success": 0, "failed": 0, "skipped": 0}

    # Process each category
    for category in SELECTED_CATEGORIES:
        category_raw = raw_dir / category

        if not category_raw.exists():
            continue

        # Create output directories
        (audio_dir / category).mkdir(parents=True, exist_ok=True)
        (image_dir / category).mkdir(parents=True, exist_ok=True)

        # Process each MP4
        mp4_files = list(category_raw.glob("*.mp4"))

        if len(mp4_files) == 0:
            continue

        print(f"\n{category}: {len(mp4_files)} videos")

        for mp4_path in tqdm(mp4_files, desc=f"  Processing"):
            video_id = mp4_path.stem

            audio_path = audio_dir / category / f"{video_id}.wav"
            image_path = image_dir / category / f"{video_id}.jpg"

            # Skip if already processed
            if audio_path.exists() and image_path.exists():
                stats["skipped"] += 1
                continue

            # Extract
            success = extract_audio_and_frame(mp4_path, audio_path, image_path)

            if success:
                stats["success"] += 1
            else:
                stats["failed"] += 1

    print(f"\n{'='*60}")
    print("Processing Summary:")
    print(f"{'='*60}")
    print(f"Successful: {stats['success']}")
    print(f"Skipped (already done): {stats['skipped']}")
    print(f"Failed: {stats['failed']}")
    print(f"Success rate: {stats['success']/(stats['success']+stats['failed'])*100:.1f}%")

    return stats

# RUN IT
print("Extracting audio and images from all MP4 files...")
print("This will take ~15-20 minutes for 1,269 videos\n")

processing_stats = process_all_videos()

print("\n✓ Audio and image extraction complete!")
print(f"✓ Location: {BASE_DIR}/processed/")

Extracting audio and images from all MP4 files...
This will take ~15-20 minutes for 1,269 videos


Processing all videos...


playing piano: 80 videos


  Processing:   0%|          | 0/80 [00:00<?, ?it/s]


playing acoustic guitar: 80 videos


  Processing:   0%|          | 0/80 [00:00<?, ?it/s]


dog barking: 80 videos


...


hammering nails: 80 videos


  Processing:   0%|          | 0/80 [00:00<?, ?it/s]


stream burbling: 80 videos


  Processing:   0%|          | 0/80 [00:00<?, ?it/s]


Processing Summary:
Successful: 1269
Skipped (already done): 0
Failed: 0
Success rate: 100.0%

✓ Audio and image extraction complete!
✓ Location: /content/drive/MyDrive/spectralbridge_data/processed/


In [8]:
# ============================================================
# Cell 9: Verify Extracted Data
# ============================================================
from pathlib import Path
import numpy as np
import soundfile as sf
from PIL import Image

print("="*60)
print("DATA VERIFICATION")
print("="*60)

audio_dir = Path(BASE_DIR) / "processed" / "audio"
image_dir = Path(BASE_DIR) / "processed" / "images"

total_audio = 0
total_images = 0
paired = 0

print("\nCategory-wise counts:")
print("-"*60)

for category in SELECTED_CATEGORIES:
    audio_files = set(f.stem for f in (audio_dir / category).glob("*.wav")) if (audio_dir / category).exists() else set()
    image_files = set(f.stem for f in (image_dir / category).glob("*.jpg")) if (image_dir / category).exists() else set()

    # Check pairing
    paired_files = audio_files & image_files

    print(f"{category:30s}: {len(audio_files):3d} audio | {len(image_files):3d} images | {len(paired_files):3d} paired")

    total_audio += len(audio_files)
    total_images += len(image_files)
    paired += len(paired_files)

print("-"*60)
print(f"{'TOTAL':30s}: {total_audio:3d} audio | {total_images:3d} images | {paired:3d} paired")
print("="*60)

if total_audio == total_images == paired:
    print("\n✅ PERFECT! All files are paired!")
else:
    print(f"\n⚠️  Mismatch detected:")
    print(f"   Audio files: {total_audio}")
    print(f"   Image files: {total_images}")
    print(f"   Paired: {paired}")

# Sample verification
print("\n" + "="*60)
print("SAMPLE FILE CHECKS")
print("="*60)

sample_cat = "playing piano"
sample_files = list((audio_dir / sample_cat).glob("*.wav"))[:3]

for audio_file in sample_files:
    image_file = image_dir / sample_cat / f"{audio_file.stem}.jpg"

    # Load and check audio
    audio_data, sr = sf.read(audio_file)

    # Load and check image
    img = Image.open(image_file)

    print(f"\n{audio_file.stem}:")
    print(f"  Audio: {audio_data.shape} @ {sr}Hz ({len(audio_data)/sr:.1f}s)")
    print(f"  Image: {img.size} {img.mode}")

    # Verify specs
    assert sr == 16000, f"Wrong sample rate: {sr}"
    assert img.size == (518, 518), f"Wrong image size: {img.size}"
    assert img.mode == "RGB", f"Wrong image mode: {img.mode}"

    print(f"  ✓ Specifications correct!")

print("\n" + "="*60)
print("✅ DATA READY FOR FEATURE EXTRACTION!")
print("="*60)
print(f"\nDataset summary:")
print(f"  - {paired} audio-visual pairs")
print(f"  - {len(SELECTED_CATEGORIES)} categories")
print(f"  - Audio: 16kHz mono WAV")
print(f"  - Images: 518×518 RGB JPG")
print(f"  - Perfect alignment by filename")
print(f"\n🚀 Ready for Week 2: Feature Extraction!")

DATA VERIFICATION

Category-wise counts:
------------------------------------------------------------
playing piano                 :  80 audio |  80 images |  80 paired
playing acoustic guitar       :  80 audio |  80 images |  80 paired
dog barking                   :  80 audio |  80 images |  80 paired
cat meowing                   :  80 audio |  80 images |  80 paired
helicopter                    :  80 audio |  80 images |  80 paired
typing on computer keyboard   :  80 audio |  80 images |  80 paired
telephone bell ringing        :  70 audio |  70 images |  70 paired
fire crackling                :  80 audio |  80 images |  80 paired
ocean burbling                :  80 audio |  80 images |  80 paired
thunder                       :  79 audio |  79 images |  79 paired
raining                       :  80 audio |  80 images |  80 paired
wind noise                    :  80 audio |  80 images |  80 paired
wind rustling leaves          :  80 audio |  80 images |  80 paired
waterfall burb